In [ ]:
%pip install -q pypdf\
 langchain-community \
 langchain-classic \
 langchain-text-splitters \
 langchain-huggingface \
 sentence-transformers \
 langchain-chroma \
 langchain-groq \
 weaviate-client \
 rank_bm25

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (create_stuff_documents_chain)
from langchain_groq import ChatGroq
from google.colab import files,userdata
import torch
import weaviate
from weaviate.classes.init import Auth

/tmp/ipykernel_15876/1635605977.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
WEAVIATE_URL = userdata.get("WEAVIATE_URL")
WEAVIATE_API_KEY = userdata.get("WEAVIATE_API_KEY")
HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers={
        "X-HuggingFace-Api-Key": HF_TOKEN
    }
)

print(client.is_ready())

True


In [ ]:
loader = PyPDFLoader('/content/2005.11401v4.pdf')

data = loader.load()

In [ ]:
print(len(data))

print(data[0].page_content[:1000])

print(data[0].metadata)

19
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigated for extractive downstream 

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

chunks = splitter.split_documents(data)

In [ ]:
print("Pages:", len(data))
print("Chunks:", len(chunks))

print(chunks[0].page_content)
print(chunks[0].metadata)

Pages: 19
Chunks: 420
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/2005.11401v4.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={
        "device": device
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_hybrid_demo"
)

In [ ]:
persist_directory="./chroma_db"

In [ ]:
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
query = "What is RAG-Token?"

results = vector_retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

Result 1
RAG-Token 43.5 54.8 46.5 51.9 17.9 22.6 56.2 49.4 74.5 90.6RAG-Sequence 44.0 55.8 44.9 53.4 15.3 21.5 57.2 47.5
{'subject': '', 'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'moddate': '2021-04-13T00:48:38+00:00', 'page': 7, 'creator': 'LaTeX with hyperref', 'total_pages': 19, 'keywords': '', 'source': '/content/2005.11401v4.pdf', 'title': '', 'creationdate': '2021-04-13T00:48:38+00:00', 'page_label': '8', 'trapped': '/False', 'producer': 'pdfTeX-1.40.21'}
--------------------------------------------------
Result 2
RAG-Token-Frozen 37.8 50.1 37.1 51.1 16.7 21.7 55.9 49.4 72.9 89.4RAG-Sequence-Frozen 41.2 52.1 41.8 52.6 11.8 19.6 56.7 47.3
{'creationdate': '2021-04-13T00:48:38+00:00', 'producer': 'pdfTeX-1.40.21', 'title': '', 'keywords': '', 'subject': '', 'trapped': '/False', 'page': 7, 'moddate': '2021-04-13T00:48:38+00:00', 'source': '/content/2005.11401v4.pdf', 'author': '', 'total_pages': 19, 'pag

In [ ]:
bm25_retriever = BM25Retriever.from_documents(
    chunks,
    k=3
)

In [ ]:
query = "What is RAG-Token?"

bm25_results = bm25_retriever.invoke(query)

for i, doc in enumerate(bm25_results, start=1):
    print(f"Result {i}")
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

Result 1
options—quuestion A is better, question B is better, both are good, or neither is good.
3.4 Fact Veriﬁcation
{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/2005.11401v4.pdf', 'total_pages': 19, 'page': 4, 'page_label': '5'}
--------------------------------------------------
Result 2
ﬁrst is to classify the claim as either "Supported", "Refuted" or "Not Enough Info", which is the task
{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subje

In [ ]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever
    ],
    weights=[0.3, 0.7]
)

In [ ]:
query = "What is RAG-Token?"

hybrid_results = hybrid_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(hybrid_results, start=1):
    print(f"Rank: {i}")
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

Rank: 1
options—quuestion A is better, question B is better, both are good, or neither is good.
3.4 Fact Veriﬁcation
{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/2005.11401v4.pdf', 'total_pages': 19, 'page': 4, 'page_label': '5'}
--------------------------------------------------
Rank: 2
ﬁrst is to classify the claim as either "Supported", "Refuted" or "Not Enough Info", which is the task
{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject

In [ ]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.2,
    max_tokens=128,
    api_key=userdata.get("GROQ_API_KEY"),
)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    Answer the question using the given context.

    If you don't know the answer, say you don't know.

    Context:
    {context}
    """),
    ("human", "{input}")
])


combine_docs_chain = create_stuff_documents_chain(
    llm,
    prompt
)


rag_chain = create_retrieval_chain(
    hybrid_retriever,
    combine_docs_chain
)


response = rag_chain.invoke({
    "input": "What is RAG-Token?"
})

print(response["answer"])

Based on the provided context, **RAG-Token** is described as a standard, autoregressive seq2seq generator. Its transition probability is defined by the formula:

$$p'_{\theta}(y_i|x, y_{1:i-1}) = \sum_{z \in \text{top-}k(p(\cdot|x))} p_{\eta}(z_i|x) p_{\theta}(y_i|x, z_i, y_{1:i-1})$$

Additionally, the context lists performance metrics for RAG-Token across various datasets (e.g., 43.5, 5
